# 03. Propositional Logic


## Setup

Jalankan sel di bawah ini sekali di awal, sebelum sel mana pun yang lain.

Sel ini memasang dependensi yang diperlukan, mencari folder yang berisi
`logic.py` dan `utils.py`, lalu mengimpornya. Kalau notebook dibuka lewat Google
Colab, repo akan di-clone otomatis. Tidak ada yang perlu diubah di sini.

Environment sudah siap kalau baris terakhir output mencetak
`Check       : tt_entails(P & Q, Q) = True`.

In [ ]:
# =============================================================================
# Standard setup cell.
# Run this once, before any other cell in this notebook.
# =============================================================================
import importlib.util
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/kcv-if/Modul-Praktikum-KK-RKA-25.git"
ON_COLAB = "google.colab" in sys.modules


def ensure_dependencies():
    """Install only the packages this module actually uses."""
    required = {
        "networkx": "networkx",
        "numpy": "numpy",
        "pandas": "pandas",
        "matplotlib": "matplotlib",
        "ipywidgets": "ipywidgets",
        "PIL": "pillow",
        "pygments": "pygments",
    }
    missing = [pkg for mod, pkg in required.items() if importlib.util.find_spec(mod) is None]
    if missing:
        print("Installing:", ", ".join(missing))
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing], check=True)


def find_environment(start):
    """Locate the folder that holds logic.py and utils.py, searching upward."""
    for root in [start, *start.parents]:
        for candidate in sorted(root.rglob("logic.py")):
            if (candidate.parent / "utils.py").exists():
                return candidate.parent
        if (root / ".git").exists():
            break
    return None


ensure_dependencies()

start_dir = Path.cwd()
if ON_COLAB:
    clone_dir = Path("Modul-Praktikum-KK-RKA-25")
    if not clone_dir.exists():
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(clone_dir)], check=True)
    start_dir = clone_dir

ENV_DIR = find_environment(start_dir)
if ENV_DIR is None:
    raise RuntimeError(
        "Environment folder not found. Make sure this notebook is opened from "
        "inside the Modul-Praktikum-KK-RKA-25 repository."
    )
if str(ENV_DIR) not in sys.path:
    sys.path.insert(0, str(ENV_DIR))

import itertools
import warnings

import pandas as pd

# qpsolvers is only used by the SVM code, which this module never touches.
warnings.filterwarnings("ignore", message="no QP solver found")

from logic import *
from notebook import psource
from utils import *

print("Environment :", ENV_DIR)
print("Python      :", sys.version.split()[0])
print("Check       : tt_entails(P & Q, Q) =", tt_entails(expr("P & Q"), expr("Q")))

---
# 3.1 Syntax Logika Proposisional

**Slide 24 dan 25**

## Penjelasan

Yang perlu masuk:

- Propositional logic sebagai logika yang sangat sederhana, sesuai judul slide
  24.
- **Syntax**: aturan sentence yang diperbolehkan.
- **Proposition symbol**: proposisi yang bisa bernilai benar atau salah. Contoh
  dari slide: P, Q, R, W12, North, True, False.
- **Complex sentence**: sentence yang dibangun dari sentence lebih sederhana
  memakai tanda kurung dan konektif logika.

Perlu dijelaskan juga bahwa proposition symbol itu atomik. Tidak ada struktur di
dalamnya. `W12` cuma nama, bukan fungsi dengan argumen 1 dan 2. Ini yang jadi
keterbatasan logika proposisional dibanding first-order logic.

## Contoh penerapan

Perkenalkan kelas `Expr` dari `logic.py`:

- `Symbol('x')` untuk satu simbol
- `symbols('x, y, P, Q')` untuk beberapa sekaligus
- Atribut `op` dan `args` dari sebuah `Expr`
- Tunjukkan bahwa simbol tunggal punya `args` kosong
- Tunjukkan bahwa `Expr` bisa bersarang, contohnya `3 * f(x, y) + P(y) / 2 + 1`

Tekankan satu hal: `Expr` cuma merepresentasikan **bentuk** kalimat, mirip
abstract syntax tree. Dia tidak menentukan nilai benar atau salah. Itu urusan
semantics di sub-topik 3.3.

---
# 3.2 Logical Connectives

**Slide 26, Figure 7.7**

## Penjelasan

Yang perlu masuk, lima konektif sesuai slide:

- **Negasi** ($\neg$). Sentence seperti $\neg W_{1,3}$ disebut negasi dari
  $W_{1,3}$. Sekalian jelaskan istilah **literal**: sentence atomik disebut
  positive literal, sentence atomik yang dinegasi disebut negative literal.
- **Konjungsi** ($\land$). Bagian-bagiannya disebut conjunct.
- **Disjungsi** ($\lor$). Bagian-bagiannya disebut disjunct. Sebutkan asal
  katanya dari bahasa Latin "vel".
- **Implikasi** ($\Rightarrow$). Bagian kiri disebut premise atau antecedent,
  bagian kanan disebut conclusion atau consequent. Implikasi juga dikenal sebagai
  rule atau pernyataan if-then. Sebutkan bahwa buku lain kadang menulisnya
  sebagai $\supset$ atau $\rightarrow$.
- **Bikondisional** ($\Leftrightarrow$). Buku lain menulisnya sebagai $\equiv$.

Istilah literal, conjunct, disjunct, antecedent, dan consequent semuanya akan
dipakai lagi di Notebook 04 dan minggu depan waktu bahas CNF. Jangan dilewat.

## Contoh penerapan

Buat tabel pemetaan antara notasi buku dan input Python. Formatnya kira-kira
begini, isi sendiri:

| Operasi | Notasi Buku | Input Python | Output Python |
|---|---|---|---|
| Negasi | | | |
| Konjungsi | | | |
| ... | | | |

Lalu tunjukkan hal-hal ini:

- Python tidak mengizinkan `==>` sebagai operator, makanya ada trik `|'==>'|`.
- Fungsi `expr()` yang menerima string, jadi tidak perlu pakai trik itu, dan
  otomatis mendefinisikan simbol yang muncul.
- Jebakan presedensi: `expr('P & Q ==> P | Q')` ternyata bukan yang dimaksud
  karena `==>` punya prioritas yang sama dengan `|`. Bandingkan dengan
  `expr('(P & Q) ==> (P | Q)')`.

Jebakan presedensi ini kelihatan sepele tapi bikin hasil salah tanpa error, jadi
wajib ditunjukkan.

---
# 3.3 Semantics

**Slide 28**

## Penjelasan

Yang perlu masuk:

- **Semantics**: aturan untuk menentukan kebenaran sebuah sentence terhadap model
  tertentu.
- **Model** dalam konteks ini menetapkan nilai benar atau salah untuk **setiap**
  proposition symbol. Contoh dari slide: kalau KB memakai simbol
  $P_{1,2}$, $P_{2,2}$, dan $P_{3,1}$, salah satu model yang mungkin adalah
  $m_1 = \{P_{1,2} = false, P_{2,2} = false, P_{3,1} = true\}$.
- Aturannya diekspresikan lewat truth table.

Kaitkan balik ke Notebook 02: di sana model direpresentasikan sebagai dict dengan
key string. Di sini bentuknya sama, cuma key-nya sekarang objek `Expr`.

Tekankan kata "setiap". Model harus menetapkan nilai untuk semua simbol, tidak
boleh ada yang dikosongkan.

## Contoh penerapan

Perkenalkan `pl_true(sentence, model)`. Tampilkan source code-nya pakai
`psource()` supaya kelihatan bahwa dia cuma evaluasi rekursif biasa.

Lalu kerjakan contoh dari slide 29: dengan
$m_1 = \{P_{1,2} = false, P_{2,2} = false, P_{3,1} = true\}$, berapa nilai
$\neg P_{1,2} \land (P_{2,2} \lor P_{3,1})$?

Hitung manual dulu langkah per langkah di markdown, baru dicek dengan `pl_true`.
Urutan ini penting supaya pembaca tidak langsung percaya kode tanpa paham.

---
# 3.4 Truth Tables dan Implikasi Material

**Slide 29 sampai 31, Figure 7.8**

## Penjelasan

Yang perlu masuk:

- Truth table untuk lima konektif, sesuai Figure 7.8.
- Cara membacanya, pakai contoh dari caption slide: untuk mencari nilai
  $P \lor Q$ ketika P benar dan Q salah, cari baris di mana P benar dan Q salah,
  lalu lihat kolom $P \lor Q$.
- Baris implikasi wajib dapat porsi paling besar. $P \Rightarrow Q$ hanya
  bernilai salah ketika P benar dan Q salah. Di tiga kasus lainnya bernilai
  benar.
- Tidak ada syarat bahwa P dan Q harus punya hubungan makna. Ini yang disebut
  implikasi material.

Dua slide terakhir (30 dan 31) adalah pertanyaan jebakan dari materi:

1. Apakah "5 is odd implies Tokyo is the capital of Japan" bernilai benar?
2. Apakah "5 is even implies Sam is smart" bernilai benar?

Jawaban keduanya benar, tapi alasannya berbeda. Yang pertama karena premis dan
konklusi sama-sama benar. Yang kedua karena premisnya salah, sehingga implikasi
otomatis benar apa pun nilai konklusinya. Kasus kedua ini yang biasanya bikin
mahasiswa protes, jadi siapkan penjelasannya.

Sekalian jelaskan kenapa aturannya dibuat begitu. Bantu dengan analogi janji:
"kalau besok hujan, saya bawa payung" cuma dilanggar kalau besok hujan dan saya
tidak bawa payung. Kalau besok tidak hujan, janji itu tidak dilanggar apa pun
yang terjadi.

## Contoh penerapan

Bangkitkan ulang Figure 7.8 dengan `pl_true`, jangan disalin dari buku.
Loop semua kombinasi P dan Q pakai `itertools.product`, lalu tampilkan sebagai
DataFrame dengan kolom untuk kelima konektif. Bandingkan hasilnya dengan gambar
di slide, harus sama persis.

Lalu jawab dua pertanyaan jebakan tadi dengan kode. Untuk yang kedua, tunjukkan
bahwa hasilnya tetap benar baik ketika Sam pintar maupun tidak.

---
# Latihan Soal

Isi bagian ini dengan minimal 3 soal. Susun dari yang paling ringan.

Format tiap soal: pernyataan soal, cell kosong untuk jawaban, lalu pembahasan
yang dibungkus `<details>`.

## Soal 1

Tingkat pemahaman. Usul arah soal: kenapa "5 is even implies Sam is smart"
bernilai benar padahal Sam belum tentu pintar? Jawab dengan merujuk ke baris
mana di truth table.

## Soal 2: Diskusi Kelompok

**Slide 27**

Ini soal resmi dari slide, jadi harus masuk apa adanya. Di slide posisinya
sebelum Semantics, tapi karena semua soal dikumpulkan di akhir, dipindah ke sini.

Soal aslinya:

> Diketahui proposisi atomic $x_1$, $x_2$, $x_3$ di mana $x_i$ merupakan notasi
> untuk kalimat "pegawai i sedang bekerja". Buatlah proposisi sesuai kondisi
> berikut:
>
> 1. Salah satu dari pegawai 1, 2 dan 3 sedang bekerja.
> 2. Terdapat dua pegawai dari pegawai 1, 2, dan 3 yang sedang bekerja.
> 3. Tidak semua pegawai sedang bekerja.
>
> Jika sudah, diskusikan jawaban kelompokmu dengan kelompok lain.

Sediakan fungsi bantu yang mengenumerasi 8 model dan mengembalikan model mana
saja yang memenuhi sebuah proposisi, supaya jawaban bisa diverifikasi sendiri.

Hal yang perlu dibahas di pembahasan: soal nomor 1 ambigu. "Salah satu" bisa
dibaca sebagai "paling sedikit satu" (7 model) atau "tepat satu" (3 model).
Ambiguitas ini justru bagian dari pelajarannya, jadi jangan diselesaikan sepihak.
Nomor 2 juga punya ambiguitas serupa antara "tepat dua" dan "paling sedikit dua".

Untuk nomor 3, kesalahan yang paling sering muncul adalah menjawab
"tidak ada yang bekerja". Bandingkan jumlah modelnya supaya bedanya kelihatan.

## Soal 3

Tingkat penerapan. Usul arah soal: buktikan hukum De Morgan
$\neg(P \land Q) \equiv \neg P \lor \neg Q$ dengan truth table yang
dibangkitkan pakai `pl_true`.

## Soal 4 (opsional)

Ruang untuk soal tambahan. Usul arah: terjemahkan tiga kalimat bahasa Indonesia
ke proposisi, dengan minimal satu kalimat yang mengandung ambiguitas.

Hapus kalau tidak dipakai.